# Clustering de Pokemon -- Arquetipos por Estadisticas Base

> ⚠️ **Módulo en desarrollo activo.**
> Los pasos de estandarización, elección de k, KMeans y PCA 2D están implementados y validados.
> Integración con el dashboard: pendiente.

**Tecnica:** K-means sobre los 6 stats base estandarizados + PCA para visualizacion 2D.
**Objetivo:** descubrir arquetipos sin etiquetas previas -- aprendizaje no supervisado.
**Datos:** 1025 formas base (id < 10000), de PokeAPI via SQLite.

Flujo: EDA -> Estandarizacion -> Eleccion de k -> K-means -> PCA -> Interpretacion -> Validacion

In [1]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
print('Dependencias OK')

Dependencias OK


## 1. Carga de datos

In [2]:
db = next(
    (p / 'data' / 'pokemon.db'
     for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data' / 'pokemon.db').exists()),
    None,
)
assert db, 'Ejecuta: python scripts/build_db.py'

con   = sqlite3.connect(db)
poke  = pd.read_sql('SELECT id, name FROM pokemon WHERE id < 10000', con)
stats = pd.read_sql('SELECT * FROM pokemon_stats', con)
sp    = pd.read_sql('SELECT id, is_legendary, is_mythical, generation FROM species', con)
tipos = pd.read_sql(
    'SELECT pokemon_id, type_name FROM pokemon_types WHERE slot=1 AND pokemon_id < 10000', con)
con.close()

STAT_COLS   = ['hp','attack','defense','special-attack','special-defense','speed']
STAT_LABELS = ['HP','Ataque','Defensa','Atq. Esp.','Def. Esp.','Velocidad']

stats_w = (stats
           .pivot(index='pokemon_id', columns='stat_name', values='base_value')
           .reset_index())
stats_w.columns.name = None

df = (poke
      .merge(stats_w, left_on='id', right_on='pokemon_id', how='left').drop(columns='pokemon_id')
      .merge(sp, on='id', how='left')
      .merge(tipos, left_on='id', right_on='pokemon_id', how='left').drop(columns='pokemon_id'))

df['bst'] = df[STAT_COLS].sum(axis=1)
df['categoria'] = 'Normal'
df.loc[df['is_legendary']==1, 'categoria'] = 'Legendario'
df.loc[df['is_mythical'] ==1, 'categoria'] = 'Mitico'

print(f'Dataset: {len(df)} Pokemon')
df.head()

Dataset: 1025 Pokemon


,id,name,attack,defense,hp,special-attack,special-defense,speed,is_legendary,is_mythical,generation,type_name,bst,categoria
0,1,bulbasaur,49,49,45,65,65,45,0,0,generation-i,grass,318,Normal
1,2,ivysaur,62,63,60,80,80,60,0,0,generation-i,grass,405,Normal
2,3,venusaur,82,83,80,100,100,80,0,0,generation-i,grass,525,Normal
3,4,charmander,52,43,39,60,50,65,0,0,generation-i,fire,309,Normal
4,5,charmeleon,64,58,58,80,65,80,0,0,generation-i,fire,405,Normal


## 2. EDA -- Distribucion de stats base

Antes de clusterizar, revisamos varianzas. Si una variable tiene mucho mayor rango,
dominara el clustering sin estandarizar.

In [3]:
df_melt = df.melt(id_vars=['name'], value_vars=STAT_COLS, var_name='stat', value_name='valor')
df_melt['stat_es'] = df_melt['stat'].map(dict(zip(STAT_COLS, STAT_LABELS)))

fig = px.box(df_melt, x='stat_es', y='valor', color='stat_es',
             title='Distribucion de stats base (1025 formas base)',
             labels={'stat_es':'Stat','valor':'Valor base'},
             template='plotly_dark')
fig.update_layout(showlegend=False, height=420)
fig.show()

df[STAT_COLS].rename(columns=dict(zip(STAT_COLS,STAT_LABELS))).describe().round(1)

,HP,Ataque,Defensa,Atq. Esp.,Def. Esp.,Velocidad
count,1025.0,1025.0,1025.0,1025.0,1025.0,1025.0
mean,70.2,77.5,72.5,70.1,70.2,67.2
std,26.6,29.8,29.3,29.7,26.6,28.7
min,1.0,5.0,5.0,10.0,20.0,5.0
25%,50.0,55.0,50.0,47.0,50.0,45.0
50%,68.0,75.0,70.0,65.0,67.0,65.0
75%,85.0,100.0,90.0,90.0,86.0,88.0
max,255.0,181.0,230.0,173.0,230.0,200.0


## 3. Estandarizacion

K-means usa distancias euclidianas. `StandardScaler` centra cada variable en 0
con desviacion estandar 1, evitando que un stat domine por tener mayor rango.

In [4]:
df_clean = df.dropna(subset=STAT_COLS).copy()
X        = df_clean[STAT_COLS].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Shape: {X_scaled.shape}')
pd.DataFrame(X_scaled, columns=STAT_LABELS).describe().round(3)

Shape: (1025, 6)


,HP,Ataque,Defensa,Atq. Esp.,Def. Esp.,Velocidad
count,1025.000,1025.000,1025.000,1025.000,1025.000,1025.000
mean,-0.000,0.000,0.000,0.000,0.000,-0.000
std,1.000,1.000,1.000,1.000,1.000,1.000
min,-2.599,-2.436,-2.306,-2.027,-1.886,-2.167
25%,-0.758,-0.757,-0.769,-0.779,-0.759,-0.773
50%,-0.082,-0.085,-0.086,-0.171,-0.120,-0.076
75%,0.557,0.755,0.598,0.672,0.593,0.725
max,6.943,3.476,5.380,3.472,6.001,4.627


## 4. Eleccion de k -- Codo + Silhouette

**Curva de codo:** la inercia (WCSS) cae al aumentar k. El 'codo' indica el punto
de rendimiento decreciente -- mas clusters no compensan la complejidad adicional.

**Silhouette score:** que tan bien separados estan los clusters (1=perfecto, -1=mal).

In [5]:
k_range     = range(2, 11)
inertias    = []
silhouettes = []

for k in k_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lbl))

fig_elbow = px.line(x=list(k_range), y=inertias, markers=True,
                    labels={'x':'k','y':'Inercia (WCSS)'},
                    title='Curva de codo',
                    template='plotly_dark')
fig_elbow.update_layout(height=360)
fig_elbow.show()

fig_sil = px.line(x=list(k_range), y=silhouettes, markers=True,
                  labels={'x':'k','y':'Silhouette score'},
                  title='Silhouette score por k (mayor = mejor separacion)',
                  template='plotly_dark')
fig_sil.update_layout(height=360)
fig_sil.show()

best_k = list(k_range)[silhouettes.index(max(silhouettes))]
print(f'k con mayor silhouette: {best_k} (score={max(silhouettes):.3f})')

k con mayor silhouette: 2 (score=0.295)


## 5. K-means final (k=5)

In [6]:
K = 5  # ajustar segun las graficas de codo/silhouette

km_final           = KMeans(n_clusters=K, random_state=42, n_init=10)
df_clean['cluster'] = km_final.fit_predict(X_scaled)

print(f'Inercia final: {km_final.inertia_:.1f}')
print('\nPokemon por cluster:')
print(df_clean['cluster'].value_counts().sort_index())

Inercia final: 2973.9

Pokemon por cluster:
cluster
0    104
1    369
2    211
3    157
4    184
Name: count, dtype: int64


## 6. Visualizacion PCA 2D

PCA reduce las 6 dimensiones a 2 componentes principales.
Los ejes no son interpretables directamente, pero la distancia entre puntos
refleja similitud en el espacio original de stats.
Pasa el cursor sobre un punto para ver el nombre del Pokemon.

In [7]:
pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
df_clean['pca_x'] = coords[:, 0]
df_clean['pca_y'] = coords[:, 1]

print(f'Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.1%}, '
      f'PC2={pca.explained_variance_ratio_[1]:.1%}, '
      f'Total={sum(pca.explained_variance_ratio_):.1%}')

fig_pca = px.scatter(
    df_clean, x='pca_x', y='pca_y',
    color='cluster', color_continuous_scale='Viridis',
    hover_name='name',
    hover_data={'bst':True,'cluster':True,'pca_x':False,'pca_y':False},
    labels={'pca_x':'PC 1','pca_y':'PC 2','cluster':'Cluster'},
    title=f'K-means k={K} -- proyeccion PCA 2D (hover = nombre del Pokemon)',
    template='plotly_dark', opacity=0.75,
)
fig_pca.update_traces(marker=dict(size=5))
fig_pca.update_layout(height=560)
fig_pca.show()

Varianza explicada: PC1=44.1%, PC2=18.4%, Total=62.5%


## 7. Interpretacion de arquetipos

El heatmap muestra el promedio de cada stat en escala original por cluster.
Busca cual stat domina para etiquetar el arquetipo.

In [8]:
profiles = df_clean.groupby('cluster')[STAT_COLS].mean().round(1)
profiles['BST'] = profiles[STAT_COLS].sum(axis=1).round(1)

prof_show = profiles[STAT_COLS].rename(columns=dict(zip(STAT_COLS,STAT_LABELS)))

fig_prof = px.imshow(
    prof_show,
    text_auto='.0f',
    color_continuous_scale='YlOrRd',
    aspect='auto',
    title='Perfil medio por cluster (stat original)',
    template='plotly_dark',
)
fig_prof.update_layout(height=380, xaxis_title='Stat', yaxis_title='Cluster')
fig_prof.show()

print(profiles.to_string())

            hp  attack  defense  special-attack  special-defense  speed    BST
cluster                                                                       
0         64.3    80.9    120.2            62.6             89.2   46.7  463.9
1         50.0    53.6     50.5            48.3             48.6   49.5  300.5
2         68.8    86.4     65.4            82.1             69.2   98.6  470.5
3         88.4    81.9     86.2           112.0            104.0   82.4  554.9
4        100.1   109.7     86.1            68.4             75.1   65.1  504.5


In [ ]:
ARCHETYPE_LABELS = {
    0: "Tanque defensivo  — Defensa alta, velocidad baja",
    1: "Pokemon debil / inicio  — Stats bajos uniformes",
    2: "Velocista especial  — Velocidad y Atq. Esp. altos",
    3: "Pseudo-legendario  — Todo alto, BST maximo",
    4: "Atacante fisico  — HP y Ataque dominantes",
}

df_clean['arquetipo'] = df_clean['cluster'].map(ARCHETYPE_LABELS)

fig_pca2 = px.scatter(
    df_clean, x='pca_x', y='pca_y',
    color='arquetipo',
    hover_name='name',
    hover_data={'bst':True,'arquetipo':False,'pca_x':False,'pca_y':False},
    labels={'pca_x':'PC 1','pca_y':'PC 2','arquetipo':'Arquetipo'},
    title=f'Arquetipos de Pokemon -- K-means k={K}',
    template='plotly_dark', opacity=0.75,
)
fig_pca2.update_traces(marker=dict(size=5))
fig_pca2.update_layout(height=560, legend_title_text='Arquetipo')
fig_pca2.show()

## 8. Validacion -- distribucion de legendarios por cluster

Sin haberle dicho al modelo que un Pokemon es legendario, deberia concentrarse
en el cluster de BST alto. Si ocurre, valida que el clustering captura estructura real.

In [10]:
leg_dist = (df_clean.groupby(['cluster','categoria'])
            .size().reset_index(name='n')
            .pivot(index='cluster', columns='categoria', values='n')
            .fillna(0).astype(int))

total = leg_dist.sum(axis=1)
leg_dist['% Especiales'] = (
    (leg_dist.get('Legendario',0) + leg_dist.get('Mitico',0)) / total * 100
).round(1)

print('Distribucion de categorias por cluster:')
print(leg_dist)

fig_val = px.bar(
    leg_dist.reset_index(), x='cluster', y='% Especiales',
    title='% de legendarios+miticos por cluster (sin haberlos etiquetado al modelo)',
    labels={'cluster':'Cluster','% Especiales':'% Legendarios + Miticos'},
    template='plotly_dark', text_auto=True,
)
fig_val.update_layout(height=380)
fig_val.show()

Distribucion de categorias por cluster:
categoria  Legendario  Mitico  Normal  % Especiales
cluster                                            
0                   3       2      99           4.8
1                   2       1     366           0.8
2                   7       4     200           5.2
3                  45      14      98          37.6
4                  14       2     168           8.7


## 9. Conclusiones

El clustering sin supervision identifica arquetipos coherentes con el diseno del juego:

- Los clusters de **BST alto** concentran legendarios y pseudo-legendarios.
- Los Pokemon de inicio del juego forman un cluster de bajo BST.
- Los clusters intermedios reflejan especializaciones: velocistas, tanques, muros.

**Proxima exploracion sugerida:** incluir el tipo primario (one-hot) o la generacion
como variable adicional para ver si los tipos forman agrupaciones naturales.

El dashboard interactivo (`dashboard/pages/5_Clusters_ML.py`) permite explorar
los clusters cambiando k y seleccionando features en tiempo real.